In [0]:
%pip install confluent-kafka

In [0]:
dbutils.library.restartPython()

In [0]:
import json, random, time
from datetime import datetime, timedelta, timezone
from collections import Counter
from confluent_kafka import Producer

# ---- Settings: use the same secret names as in 01_kafka_check ----
SCOPE, K_BOOT, K_KEY, K_SECRET = "pa_clock_scope", "kafka-bootstrap", "kafka-api-key", "kafka-api-secret"
TOPIC = "pa-events"

dbutils.widgets.text("n_cases", "300", "Cases")
dbutils.widgets.text("seed", "42", "Seed")
dbutils.widgets.text("sleep_ms", "0", "Sleep ms between messages")
dbutils.widgets.dropdown("mode", "send", ["send", "dry_run"], "Mode")
N_CASES, SEED = int(dbutils.widgets.get("n_cases")), int(dbutils.widgets.get("seed"))

SLEEP_MS, MODE = int(dbutils.widgets.get("sleep_ms")), dbutils.widgets.get("mode")

# Decision windows in hours (CMS-0057-F). Keep in sync with ref.ref_pa_rules.
WINDOW_H = {"STANDARD": 168, "EXPEDITED": 72}

# Scenario mix (name, weight). Open cases are placed so alerts fire at every risk band.
SCENARIOS = [("decided_on_time", 45), ("decided_late", 8), ("withdrawn", 8),
             ("open_on_track", 15), ("open_warning", 10), ("open_critical", 8), ("open_breached", 6)]
OPEN_RANGE = {"open_on_track": (0.05, 0.70), "open_warning": (0.76, 0.89),
              "open_critical": (0.91, 0.99), "open_breached": (1.02, 1.40)}

rng = random.Random(SEED)
now = datetime.now(timezone.utc)
run_id = now.strftime("%m%d%H%M")
iso = lambda dt: dt.strftime("%Y-%m-%dT%H:%M:%SZ")

MEMBERS   = [f"M{i:05d}" for i in range(1, 201)]                       # matches ref.dim_member
PROVIDERS = [str(rng.randrange(1_000_000_000, 9_999_999_999)) for _ in range(25)]
SERVICES  = ["72148", "70553", "27447", "93306", "97110", "95810", "63030", "29881"]
REVIEWERS = [f"UM{n:02d}" for n in range(1, 9)]

def build_case(i, scenario):
    """Return [(event_time, event_dict)] for one PA case."""
    urgency = rng.choices(["STANDARD", "EXPEDITED"], [75, 25])[0]
    W = WINDOW_H[urgency]
    base = dict(case_id=f"PA-{run_id}-{i:04d}", member_id=rng.choice(MEMBERS),
                provider_npi=rng.choice(PROVIDERS), service_code=rng.choice(SERVICES), urgency=urgency)
    tf = None                                   # decision/withdraw turnaround as a fraction of the window
    if scenario == "decided_on_time": tf = rng.uniform(0.10, 0.95); elapsed = tf + rng.uniform(0.05, 0.8)
    elif scenario == "decided_late":  tf = rng.uniform(1.05, 1.50); elapsed = tf + rng.uniform(0.05, 0.5)
    elif scenario == "withdrawn":     tf = rng.uniform(0.10, 0.80); elapsed = tf + rng.uniform(0.05, 0.5)
    else:                             elapsed = rng.uniform(*OPEN_RANGE[scenario])
    submitted = now - timedelta(hours=elapsed * W)
    evs = [("SUBMITTED", submitted, None)]
    if (tf if tf is not None else elapsed) > 0.4 and rng.random() < 0.35:   # info-request round trip
        evs += [("INFO_REQUESTED", submitted + timedelta(hours=0.15 * W), None),
                ("INFO_RECEIVED",  submitted + timedelta(hours=0.30 * W), None)]
    if scenario.startswith("decided"):
        evs.append(("DECIDED", submitted + timedelta(hours=tf * W),
                    rng.choices(["APPROVED", "DENIED", "PARTIAL"], [70, 20, 10])[0]))
    elif scenario == "withdrawn":
        evs.append(("WITHDRAWN", submitted + timedelta(hours=tf * W), None))
    reviewer = rng.choice(REVIEWERS)
    return [(ts, {**base, "event_type": et, "event_ts": iso(ts), "decision": dec,
                  "reviewer_id": reviewer if et in ("INFO_REQUESTED", "DECIDED") else None})
            for et, ts, dec in evs]

# ---- Build all events ----
names, weights = zip(*SCENARIOS)
mix, items = Counter(), []                      # items: (send_time, event)
for i in range(1, N_CASES + 1):
    sc = rng.choices(names, weights)[0]; mix[sc] += 1
    case_events = build_case(i, sc)
    late_first = len(case_events) >= 2 and rng.random() < 0.08   # first event arrives late => out-of-order
    for j, (ts, ev) in enumerate(case_events):
        delay = timedelta(hours=0.5 * WINDOW_H[ev["urgency"]]) if (late_first and j == 0) else timedelta(0)
        items.append((ts + delay, ev))
items.sort(key=lambda x: x[0])
events = [ev for _, ev in items]
for n, ev in enumerate(events, 1):
    ev["event_id"] = f"e-{run_id}-{n:06d}"

# ---- Add realistic mess: ~5% duplicates, ~2% malformed ----
msgs, dup_n = [], 0
for ev in events:
    msgs.append(ev)
    if rng.random() < 0.05: msgs.append(dict(ev)); dup_n += 1
bad_n = max(2, int(len(msgs) * 0.02))
for _ in range(bad_n):
    b = dict(rng.choice(events)); b["event_id"] = f"e-{run_id}-bad{rng.randrange(10**6)}"
    if rng.random() < 0.5: b["case_id"] = None
    else: b["event_type"] = "UNKNOWN_TYPE"
    msgs.insert(rng.randrange(len(msgs)), b)

# ---- Produce ----
ok = errs = 0
def on_delivery(err, msg):
    global ok, errs
    if err is None: ok += 1
    else: errs += 1

if MODE == "send":
    producer = Producer({
        "bootstrap.servers": dbutils.secrets.get(SCOPE, K_BOOT),
        "security.protocol": "SASL_SSL", "sasl.mechanisms": "PLAIN",
        "sasl.username": dbutils.secrets.get(SCOPE, K_KEY),
        "sasl.password": dbutils.secrets.get(SCOPE, K_SECRET),
        "linger.ms": 20})
    for m in msgs:
        producer.produce(TOPIC, key=(m.get("case_id") or "NO_CASE"), value=json.dumps(m), on_delivery=on_delivery)
        producer.poll(0)
        if SLEEP_MS: time.sleep(SLEEP_MS / 1000)
    producer.flush(60)

print(f"Mode={MODE} | cases={N_CASES} | messages={len(msgs)} (duplicates={dup_n}, malformed={bad_n})")
print("Scenario mix:", dict(mix))
print(f"Delivered={ok} | errors={errs}")
print("Sample:", json.dumps(msgs[0]))